In [5]:
import psycopg2

conn = psycopg2.connect(
        host="localhost",
        database="kenexaihackathon",
        user="postgres",
        password="09092002",
        port="5432"
    )

cur = conn.cursor()

In [6]:
print("Loading Auvik alerts...")

cur.execute("""
INSERT INTO silver.alerts (
    alert_id,
    source_system,
    device_name,
    device_identifier,
    organization_name,
    alert_type,
    severity,
    alert_message,
    occurred_at,
    ingestion_time
)

SELECT
    alert_id,
    'Auvik',
    entity_name,
    entity_id,
    company_name,
    alert_name,
    alert_severity_string,
    alert_description,
    alert_date,
    ingestion_time

FROM bronze.auvik_alerts_raw
ON CONFLICT (alert_id) DO NOTHING;
""")

print("Loading Meraki alerts...")

cur.execute("""
INSERT INTO silver.alerts (
    alert_id,
    source_system,
    device_name,
    device_identifier,
    organization_name,
    network_name,
    alert_type,
    severity,
    alert_message,
    occurred_at,
    ingestion_time
)

SELECT
    alert_id,
    'Meraki',
    device_name,
    device_serial,
    organization_name,
    network_name,
    alert_type,
    alert_level,
    check_name,
    occurred_at,
    ingestion_time

FROM bronze.meraki_alerts_raw
ON CONFLICT (alert_id) DO NOTHING;
""")

print("Loading N-Central alerts...")

cur.execute("""
INSERT INTO silver.alerts (
    alert_id,
    source_system,
    device_name,
    device_identifier,
    organization_name,
    alert_type,
    severity,
    alert_message,
    occurred_at,
    ingestion_time
)

SELECT
    COALESCE(
        active_notification_trigger_id::TEXT,
        md5(device_name || time_of_state_change::TEXT)
    ),
    'NCentral',
    device_name,
    device_uri,
    customer_name,
    affected_service,
    qualitative_new_state,
    quantitative_new_state,
    time_of_state_change,
    ingestion_time

FROM bronze.ncentral_alerts_raw
ON CONFLICT (alert_id) DO NOTHING;
""")

conn.commit()

cur.close()
conn.close()

print("Silver layer ingestion completed.")

Loading Auvik alerts...
Loading Meraki alerts...
Loading N-Central alerts...
Silver layer ingestion completed.
